### **Extraer el `titulo`, `autor`, `duración` y `enlace del libro` de múltiples páginas y exportarlos a un archivo `CSV`**

Vamos a trabajar en el enlace:

https://www.audible.com/search/

**`Al momento de extraer multiples paginas dentro de un sitio se recomienda utilizar Scrapy sobre Selenium, porque es MUCHO MÁS RÁPIDO.`**

#### **Crear la spider**

Vamos a crear la spider:

<center><img src="https://i.postimg.cc/X7FP964h/ws-442.png"></center>

Luego, debemos editar el archivo `audible_dos.py` que se crea, con el código que se muestra abajo.

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/sX7MB5jq/ws-432.png"></center>

Localizamos cada uno de los libros:

<center><img src="https://i.postimg.cc/j2TLyCC5/ws-434.png"></center>

Localizamos el `nombre` de cada audiolibro:

<center><img src="https://i.postimg.cc/wjjRWt2R/ws-435.png"></center>

Localizamos el `autor` de cada audiolibro:

<center><img src="https://i.postimg.cc/q7thT99k/ws-437.png"></center>

Localizamos el `duración` de cada audiolibro:

<center><img src="https://i.postimg.cc/0yXMh8XW/ws-438.png"></center>

Localizamos el `enlace` de cada audiolibro:

<center><img src="https://i.postimg.cc/xdWkT62L/ws-436.png"></center>

Localizamos la `barra de paginación`:

<center><img src="https://i.postimg.cc/7hzK1S87/ws-443.png"></center>

Localizamos el `enlace` del `botón de Siguiente` de la barra de paginación:

<center><img src="https://i.postimg.cc/L4gQMFYX/ws-444.png"></center>

#### **Código**

Así que en la condición `if` que escribimos significa que si hay una "página siguiente" o si la "página siguiente" existe, vaya a la "página siguiente". Pero si no hay una página "página siguiente", a continuación, detener el proceso de scraping. Esto es mucho más fácil que el bucle while que se utilizaría en Selenium.

In [ ]:
import scrapy


class AudibleDosSpider(scrapy.Spider):
    name = "audible_dos"
    allowed_domains = ["www.audible.com"]
    start_urls = ["https://www.audible.com/search/"]

    def parse(self, response):
        # Extracción de nombres de paises y su población
        productos = response.xpath('//li[contains(@class, "productListItem")]')

        for audiolibro in productos:
            titulo = audiolibro.xpath('.//h3/a/text()').get()
            # Obtenemos un 'Enlace relativo' --> /world-population/india-population/
            autor = audiolibro.xpath('.//li[contains(@class, "authorLabel")]/span/a/text()').get()
            duracion = audiolibro.xpath('.//li[contains(@class, "runtimeLabel")]/span/text()').get()
            # Obtenemos un 'Enlace relativo' --> /es_US/pd/Bondsfungi-Audiolibro/B0CZY9L58G?qid=1712613729&sr=1-1&ref_pageloadid=not_applicable&ref=a_search_c3_lProduct_1_1&pf_rd_p=83218cca-c308-412f-bfcf-90198b687a2f&pf_rd_r=4KVRZFDHGXG64G59NVEJ&pageLoadId=eHqNzIJEsQWCH0nx&creativeId=0d6f6720-f41c-457e-a42b-8c8dceb62f2c
            enlace = audiolibro.xpath('.//h3/a/@href').get()

            # Devuelve los datos extraidos
            # Vamos a devolver un 'Enlace absoluto'
            yield {
                'titulo':titulo,
                'autor': autor,
                'duracion': duracion,
                'enlace': response.urljoin(enlace), # https://www.audiolibro.com/es_US/pd/Bondsfungi-Audiolibro/B0CZY9L58G?qid=1712613729&sr=1-1&ref_pageloadid=not_applicable&ref=a_search_c3_lProduct_1_1&pf_rd_p=83218cca-c308-412f-bfcf-90198b687a2f&pf_rd_r=4KVRZFDHGXG64G59NVEJ&pageLoadId=eHqNzIJEsQWCH0nx&creativeId=0d6f6720-f41c-457e-a42b-8c8dceb62f2c
            }
        
        # Obtener la barra de paginacion 
        paginacion = response.xpath('//ul[contains(@class, "pagingElements")]')
        pagina_siguiente_url = paginacion.xpath('.//span[contains(@class, "nextButton")]/a/@href').get()

        # Ir al enlace de pagina_siguiente_url:
        if pagina_siguiente_url:
            yield response.follow(url=pagina_siguiente_url, callback=self.parse,
                                  headers={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'})



#### **Ejecución**

Nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl audible_dos

<center><img src="https://i.postimg.cc/Cxs66CZs/ws-445.png"></center>

Si nos fijamos bien, solo nos devolvió el scrapping de 20 elementos, es decir, de la primera página. Me fije que aparecia el mensaje:

**`DEBUG: Forbidden by robots.txt`**

Y aparecia junto al mensaje la url de la siguiente página:

<center><img src="https://i.postimg.cc/LXwyZN72/ws-446.png"></center>

Por tanto, para solucionar este problema, desde el archivo `settings.py` modificamos el valor de `ROBOTSTXT_OBEY`, de tener un valor `True` por defecto lo cambiamos a `False`:

In [ ]:
ROBOTSTXT_OBEY = False

Y obtuvimos la totalidad de los audiolibros:

<center><img src="https://i.postimg.cc/HxVZxf66/ws-447.png"></center>

Podriamos exportar toda esta data en un archivo `CSV` escribiendo el siguiente comando:

In [ ]:
scrapy crawl audible_dos -o audible_dos.csv